In [0]:
storage_account = "<your_storage_account>"

bronze_path = (
    f"abfss://bronze@{storage_account}.dfs.core.windows.net/"
    "customer_transactions/"
)

bronze_path

'abfss://bronze@<your_storage_account>.dfs.core.windows.net/customer_transactions/'

In [0]:
hconf = spark.sparkContext._jsc.hadoopConfiguration()

print(
    hconf.get(
        "fs.azure.account.key.<your_storage_account>.dfs.core.windows.net"
    )
)


None


In [0]:
display(
    dbutils.fs.ls(bronze_path)
)

path,name,size,modificationTime
abfss://bronze@.dfs.core.windows.net/customer_transactions/day1.csv,day1.csv,32014194,1785216940000
abfss://bronze@.dfs.core.windows.net/customer_transactions/day2.csv,day2.csv,32199629,1785216939000
abfss://bronze@.dfs.core.windows.net/customer_transactions/day3.csv,day3.csv,32623903,1785216939000


Import functions

In [0]:
from pyspark.sql import functions as F

Reading files

In [0]:
day1_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(bronze_path + "day1.csv")
    .withColumn(
        "load_date",
        F.lit("2026-01-01")
    )
)

In [0]:
day2_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(bronze_path + "day2.csv")
    .withColumn(
        "load_date",
        F.lit("2026-01-02")
    )
)

In [0]:
day3_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(bronze_path + "day3.csv")
    .withColumn(
        "load_date",
        F.lit("2026-01-03")
    )
)

Combining datasets

In [0]:
customer_transactions_df = (
    day1_df
    .unionByName(day2_df)
    .unionByName(day3_df)
)

In [0]:
customer_transactions_df.count()

300000

Inspect Schema

In [0]:
customer_transactions_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- customer_age: double (nullable = true)
 |-- customer_gender: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- loyalty_tier: string (nullable = true)
 |-- country: string (nullable = true)
 |-- state: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- transaction_id: string (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- shipment_date: timestamp (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- sales_channel: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- di

Define the Silver Path

In [0]:
display(
    dbutils.fs.ls(
        "abfss://silver@<your_storage_account>.dfs.core.windows.net/"
    )
)

path,name,size,modificationTime
abfss://silver@.dfs.core.windows.net/customer_transactions/,customer_transactions/,0,0


In [0]:
silver_path = (
    "abfss://silver@<your_storage_account>.dfs.core.windows.net/"
    "customer_transactions/"
)

Write the Delta Table

In [0]:
(
    customer_transactions_df.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("load_date")
    .save(silver_path)
)

In [0]:
display(
    dbutils.fs.ls(silver_path)
)

path,name,size,modificationTime
abfss://silver@.dfs.core.windows.net/customer_transactions/_delta_log/,_delta_log/,0,1785234005000
abfss://silver@.dfs.core.windows.net/customer_transactions/load_date=2026-01-01/,load_date=2026-01-01/,0,1785234008000
abfss://silver@.dfs.core.windows.net/customer_transactions/load_date=2026-01-02/,load_date=2026-01-02/,0,1785234010000
abfss://silver@.dfs.core.windows.net/customer_transactions/load_date=2026-01-03/,load_date=2026-01-03/,0,1785234011000


Silver Table

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS customer_transactions_silver
USING DELTA
LOCATION 'abfss://silver@<your_storage_account>.dfs.core.windows.net/customer_transactions/'
""")

DataFrame[]

In [0]:
%sql
SELECT *
FROM customer_transactions_silver
LIMIT 10;

customer_id,customer_name,customer_age,customer_gender,customer_segment,loyalty_tier,country,state,city,region,product_id,product_name,product_category,subcategory,brand,transaction_id,order_date,shipment_date,payment_method,sales_channel,currency,quantity,unit_price,discount,tax,shipping_cost,total_sales,profit,profit_margin,warehouse,delivery_days,supplier,inventory_level,return_flag,website_visits,cart_size,coupon_used,session_duration,load_date
CUST100125,David Robinson,47.0,Male,Small Business,Gold,Germany,Hesse,Frankfurt,Europe,PRD20740,Zentek Cameras 938,Electronics,Cameras,Zentek,TXN21000000,2026-02-13T16:28:00Z,2026-02-19T16:28:00Z,Credit Card,In-Store,EUR,3,1163.31,0.134,275.79,11.95,3310.02,554.2,0.1674,WH-APAC-02,6.0,Anchor Wholesale,344.0,0,3,5,0.0,6.8,2026-01-02
CUST114825,Karen Jackson,50.0,Female,Small Business,Silver,Canada,Quebec,Quebec City,North America,PRD20313,DriveMax Accessories 943,Automotive,Accessories,DriveMax,TXN21000001,2026-02-10T15:56:00Z,2026-02-14T15:56:00Z,Gift Card,In-Store,CAD,3,257.66,0.08,107.5,13.34,831.98,244.88,0.2943,WH-APAC-01,4.0,Pinnacle Supply Co,514.0,0,9,6,0.0,10.6,2026-01-02
CUST115961,Wei Rivera,53.0,Female,Consumer,Bronze,Germany,Hesse,Frankfurt,Europe,PRD21074,Domus Appliances 919,Home & Kitchen,Appliances,Domus,TXN21000002,2026-02-01T00:34:00Z,2026-02-06T00:34:00Z,Digital Wallet,Online,EUR,4,763.76,0.19,284.77,1.61,2760.96,727.1,0.2634,WH-LATAM-01,5.0,Meridian Traders,312.0,0,4,4,1.0,10.1,2026-01-02
CUST114068,Sarah Hill,59.0,Male,Consumer,Platinum,Vietnam,Ho Chi Minh,Ho Chi Minh City,Asia Pacific,PRD20919,KidJoy Board Games 223,Toys & Games,Board Games,KidJoy,TXN21000003,2026-02-15T03:03:00Z,2026-02-21T03:03:00Z,Credit Card,Online,VND,4,45.46,0.038,23.26,10.97,209.16,84.16,0.4024,WH-NA-01,6.0,Meridian Traders,154.0,0,5,7,0.0,7.7,2026-01-02
CUST123541,Linda Rivera,30.0,Female,Home Office,Bronze,India,Delhi,New Delhi,Asia Pacific,PRD20962,Zentek Security Cameras 608,Smart Home,Security Cameras,Zentek,TXN21000004,2026-02-18T21:18:00Z,2026-02-22T21:18:00Z,Bank Transfer,Call Center,INR,4,155.67,0.033,89.38,3.45,694.96,213.58,0.3073,WH-LATAM-01,4.0,Meridian Traders,588.0,0,3,5,0.0,13.4,2026-01-02
CUST107872,Valentina Brown,40.0,Male,Consumer,Bronze,UK,Wales,Cardiff,Europe,PRD20558,DriveMax Electronics 992,Automotive,Electronics,DriveMax,TXN21000005,2026-02-20T23:47:00Z,2026-02-28T23:47:00Z,Credit Card,In-Store,GBP,4,202.64,0.092,138.95,7.03,881.97,247.06,0.2801,WH-NA-01,8.0,Meridian Traders,166.0,0,8,5,0.0,6.9,2026-01-02
CUST104347,Yuki Thompson,42.0,Female,Consumer,Gold,Australia,New South Wales,Sydney,Asia Pacific,PRD21360,Harvest Beverages 141,Grocery,Beverages,Harvest,TXN21000006,2026-02-20T11:41:00Z,2026-02-26T11:41:00Z,Gift Card,Marketplace,AUD,3,33.31,0.107,11.47,3.29,104.0,16.49,0.1586,WH-EU-02,6.0,GlobalSource Ltd,442.0,0,3,5,0.0,8.4,2026-01-02
CUST121588,Chen Martin,48.0,Male,Home Office,Bronze,France,Ile-de-France,Paris,Europe,PRD20220,TechNova Cameras 726,Electronics,Cameras,TechNova,TXN21000007,2026-02-20T07:47:00Z,2026-02-26T07:47:00Z,Digital Wallet,In-Store,EUR,7,532.77,0.413,218.48,6.89,2414.52,-448.27,-0.1857,WH-APAC-02,6.0,Vertex Distribution,394.0,0,3,7,1.0,10.7,2026-01-02
CUST100563,Lucas Wilson,40.0,Male,Corporate,Silver,Mexico,Jalisco,Guadalajara,Latin America,PRD20306,Domus Furniture 838,Home & Kitchen,Furniture,Domus,TXN21000008,2026-02-04T00:19:00Z,2026-02-08T00:19:00Z,Debit Card,In-Store,MXN,7,46.49,0.084,31.97,6.69,336.75,111.95,0.3324,WH-EU-02,4.0,Vertex Distribution,569.0,0,6,8,0.0,4.1,2026-01-02
CUST103274,Joseph Jackson,48.0,Female,Home Office,Silver,USA,California,San Diego,North America,PRD21028,Domus Furniture 241,Home & Kitchen,Furniture,Domus,TXN21000009,2026-02-22T16:01:00Z,2026-02-25T16:01:00Z,Credit Card,Online,USD,6,560.97,0.122,207.45,8.07,3170.71,1029.94,0.3248,WH-APAC-01,3.0,Meridian Traders,607.0,0,4,9,0.0,19.0,2026-01-02


Validate the Silver Layer

In [0]:
%sql
SELECT COUNT(*)
FROM customer_transactions_silver;

count(1)
300000


In [0]:
%sql
SELECT 
    load_date,
    COUNT(*) AS records
FROM customer_transactions_silver
GROUP BY load_date
ORDER BY load_date;

load_date,records
2026-01-01,100000
2026-01-02,100000
2026-01-03,100000
